# Supplementary Table 1

Candidate-clustering search sensitivity in 15 CAMPI acquisitions: for each acquisition, the unclustered search and six MMseqs2/Linclust identity thresholds (99, 97, 95, 90, 75 and 50%), 105 rows. This notebook rebuilds the table from `source_data/STab1/`, compares it with the table as produced, repeats the producer's checks and prints the medians quoted in the Results.

Run from the repository root: `python paper.py build STab1`.

In [1]:
from pathlib import Path
import sys
UNIT = Path.cwd()                 # the notebook runs with its own directory as cwd
REPO = UNIT.parents[1]
sys.path[:0] = [str(REPO), str(UNIT)]
SD = REPO / "source_data" / UNIT.name
OUT = REPO / "build" / UNIT.name   # rendered panels; gitignored
OUT.mkdir(parents=True, exist_ok=True)
from style import fl_style as fl
import panels
fl.init()

## Table

105 rows, one per acquisition (n = 15 CAMPI acquisitions) and arm (unclustered plus six identity thresholds). Supports "all 105 acquisition–arm configurations and paired counts are supplied in Supplementary Table 1" (v28d line 695) and the Results sentence on clustering (v28d line 89). No test; counts and paired ratios relative to the same acquisition's unclustered search.

In [2]:
import pandas as pd
d = panels.load(SD)
table = panels.build_table(d)
tsv, xlsx = panels.write_table(table, OUT)
print(table.shape)
pd.set_option("display.max_columns", 30)
table.head(14)

(105, 21)


,Acquisition,Arm,Linclust minimum sequence identity (%),Searched protein entries (n),Protein entry ratio (clustered/unclustered),Accepted peptides (n),Unclustered accepted peptides (n),Peptide count ratio (clustered/unclustered),Peptide reduction (%),Peptides lost vs unclustered (n),Peptides gained vs unclustered (n),Unclustered peptide identities retained (fraction),Reference-database peptides (n),Reference-database peptides recovered (n),Reference-database peptide recovery (fraction),Compatible KOs (n),Unclustered compatible KOs (n),Unclustered KOs retained (n),KOs lost vs unclustered (n),KOs gained vs unclustered (n),KO recovery (fraction)
0,S01,unclustered,<NA>,3423,1.000000,21467,21467,1.000000,0.000000,0,0,1.000000,23474,17627,0.750916,1252,1252,1252,0,0,1.000000
1,S01,cluster_099,99,3491,1.019866,21043,21467,0.980249,1.975125,1070,646,0.950156,23474,17289,0.736517,1246,1252,1239,13,7,0.989617
2,S01,cluster_097,97,3601,1.052001,20717,21467,0.965063,3.493735,1618,868,0.924628,23474,17102,0.728551,1240,1252,1237,15,3,0.988019
3,S01,cluster_095,95,3680,1.075080,20415,21467,0.950995,4.900545,2049,997,0.904551,23474,16985,0.723566,1236,1252,1230,22,6,0.982428
4,S01,cluster_090,90,3930,1.148116,19849,21467,0.924628,7.537150,2655,1037,0.876322,23474,16610,0.707591,1224,1252,1217,35,7,0.972045
5,S01,cluster_075,75,4177,1.220275,17520,21467,0.816136,18.386360,5010,1063,0.766619,23474,15253,0.649783,1215,1252,1197,55,18,0.956070
6,S01,cluster_050,50,4202,1.227578,17927,21467,0.835096,16.490427,4614,1074,0.785065,23474,15518,0.661072,1225,1252,1202,50,23,0.960064
7,S02,unclustered,<NA>,3279,1.000000,18965,18965,1.000000,0.000000,0,0,1.000000,20774,15482,0.745258,1194,1194,1194,0,0,1.000000
8,S02,cluster_099,99,3327,1.014639,15338,18965,0.808753,19.124703,4259,632,0.775428,20774,13114,0.631270,1120,1194,1115,79,5,0.933836
9,S02,cluster_097,97,3419,1.042696,18035,18965,0.950962,4.903770,1637,707,0.913683,20774,14850,0.714836,1185,1194,1178,16,7,0.986600


## Producer checks

The earlier 45 rows (unclustered, 97%, 99%) and the 60 rows added for 95/90/75/50% must equal the producer table under the producer's column mapping; an independent recount from the Sage result files must agree on all 105 rows; all 90 clustered rows must lie below their unclustered baseline.

In [3]:
chk, ok = panels.checks(d)
chk.to_csv(OUT / "CHECKS.tsv", sep="\t", index=False)
print(chk.to_string(index=False))
print("ALL_CHECKS_PASS" if ok else "CHECK_FAILURE")
assert ok

                        check    rows_or_value expected_or_value missing int_mismatches float_not_bit_identical float_max_abs_diff
   current_45_rows_vs_mapping               45                45       0              0                      29           1.11e-16
       new_60_rows_vs_mapping               60                60       0              0                       0                  0
 independent_recount_vs_table              105               105       0              0                                           
           median_cluster_050 reduction 8.774%         KO 96.40%                                                                  
           median_cluster_075 reduction 7.226%         KO 96.53%                                                                  
           median_cluster_090 reduction 4.149%         KO 97.77%                                                                  
           median_cluster_095 reduction 2.457%         KO 98.41%                   

## Checks against the table as produced and recomputed numbers

The TSV must be byte-identical to `expected/Supplementary_Table_1.tsv`; the XLSX is compared cell by cell (its bytes carry a timestamp); the check table must be byte-identical to `expected/CHECKS.tsv`. Receipts go to `checks/`. Then every number in `NUMBERS.tsv` is printed in order.

In [4]:
import hashlib, json
def sha(p): return hashlib.sha256(Path(p).read_bytes()).hexdigest()
exp = UNIT / "expected"
receipt = {
    "tsv": {"new_sha256": sha(tsv), "expected_sha256": sha(exp / "Supplementary_Table_1.tsv")},
    "checks_tsv": {"new_sha256": sha(OUT / "CHECKS.tsv"), "expected_sha256": sha(exp / "CHECKS.tsv")},
    "xlsx": panels.compare_xlsx_values(xlsx, exp / "Supplementary_Table_1.xlsx"),
}
receipt["tsv"]["identical"] = receipt["tsv"]["new_sha256"] == receipt["tsv"]["expected_sha256"]
receipt["checks_tsv"]["identical"] = receipt["checks_tsv"]["new_sha256"] == receipt["checks_tsv"]["expected_sha256"]
receipt["xlsx"]["expected_sha256"] = sha(exp / "Supplementary_Table_1.xlsx")
receipt["result"] = "IDENTICAL" if all(receipt[k]["identical"] for k in ("tsv", "checks_tsv", "xlsx")) else "DIFFERENT"
(UNIT / "checks").mkdir(exist_ok=True)
(UNIT / "checks" / "table.json").write_text(json.dumps(receipt, indent=2) + "\n")
print(json.dumps(receipt, indent=2))

med = panels.stat_medians(d)
print("NUMBERS.tsv order")
print("89  acquisitions reduced at every threshold:", panels.stat_acquisitions_reduced_at_every_threshold(d))
for arm in ["cluster_099", "cluster_097", "cluster_095", "cluster_090", "cluster_075", "cluster_050"]:
    print(f"89  median peptide reduction {arm}: {repr(float(med.loc[arm, 'red']))} (n={int(med.loc[arm, 'n'])})")
lo, hi = panels.stat_ko_median_range_pct(d)
print("89  lowest median KO recovery %:", repr(lo))
print("89  highest median KO recovery %:", repr(hi))
print("475 acquisitions:", panels.stat_n_acquisitions(d))
print("695 acquisition-arm configurations:", panels.stat_n_rows(d))
print("(check) clustered rows below baseline:", panels.stat_rows_below_baseline(d))

{
  "tsv": {
    "new_sha256": "4de2e9be55a9ec95587edb24a349803651ccccbce921f3ea1e5b414b0761c085",
    "expected_sha256": "4de2e9be55a9ec95587edb24a349803651ccccbce921f3ea1e5b414b0761c085",
    "identical": true
  },
  "checks_tsv": {
    "new_sha256": "77af7022363ba41a7359ddd63622071ec077f03659915f6c3e6ec91e3ad01cce",
    "expected_sha256": "77af7022363ba41a7359ddd63622071ec077f03659915f6c3e6ec91e3ad01cce",
    "identical": true
  },
  "xlsx": {
    "sheets": [
      [
        "Supplementary_Table_1"
      ],
      [
        "Supplementary_Table_1"
      ]
    ],
    "cells_compared": 2226,
    "cells_differing": 0,
    "freeze_panes": [
      "C2",
      "C2"
    ],
    "identical": true,
    "expected_sha256": "1bd83133aeca8a1cbe0b56970ad8c8a3e57e3a5b9a9e3a616e9dff39f7bdfd5e"
  },
  "result": "IDENTICAL"
}
NUMBERS.tsv order
89  acquisitions reduced at every threshold: 15
89  median peptide reduction cluster_099: 0.8709800918836108 (n=15)
89  median peptide reduction cluster_097: 2.1